# Clinical MCQ: fine-tuned model + RAG vs. base model

Final comparison for the writeup: **zero-shot base Phi-3.5-mini** vs. **the full pipeline**
(LoRA fine-tuned on board-exam Q&A + FAISS retrieval over the knowledge base). Two numbers,
same eval sets, same harness — that delta is the headline result.

**Requires Internet = On.** Downloads the base model, MiniLM, and installs `unsloth`/`faiss`/
`sentence-transformers` at runtime — same setup as the training notebook. If this notebook is
attached to a competition that has locked internet (common after a submission), none of this
will run; use the offline build instead (fp16 + plain transformers/peft + numpy cosine sim +
locally-attached model weights) rather than fighting the lock.

**What the harness gets right that the original notebook didn't:**

1. **Answer extraction was reading the prompt, not the model.** `tokenizer.decode(output[0])`
   returns *prompt + completion*, and the regex `\b([A-D])\b` matched the first letter in the
   formatted options block, not the generation. Fixed by scoring log-probabilities of A/B/C/D
   directly instead of generating and regexing.
2. **Prompt format mismatch.** Training uses the Phi-3 chat template
   (`<|system|>/<|user|>/<|assistant|>` with `### Question:` headers); inference now reproduces
   it exactly instead of a plain-text prompt the adapter never saw.
3. **Contamination check** between train / KB / eval sets, reported explicitly so the delta is
   defensible.
4. **RAG context includes the KB's answer/output field**, not just the question-like
   `instruction` text.
5. **Results persisted to `results/`** so numbers survive the session.

> **Turn the GPU on and Internet on.** Settings → Accelerator → GPU T4, Internet → On.

In [ ]:
!pip install -q faiss-cpu sentence-transformers

In [ ]:
%%capture
import os

if "COLAB_" not in "".join(os.environ.keys()):
    # EXACT pins, not a range. An earlier version of this cell used
    # transformers>=4.56.1,<5.0 (a range within unsloth\'s declared support
    # window). That still broke: `pip install -U` re-resolves to whatever\'s
    # NEWEST in the range every time it runs, and a later run landed on
    # transformers 4.57.6, which shipped configuration_cohere2.py importing
    # layer_type_validation from configuration_utils.py -- a symbol that
    # SPECIFIC release didn\'t actually have. An internal inconsistency inside
    # transformers itself, not a cross-package conflict a range can dodge.
    #
    # These exact versions are copied from finetune-lora.ipynb\'s successful
    # training run (its own version-check cell, post-install) -- a combination
    # already proven to import and run cleanly, not re-derived from metadata.
    # If retraining ever moves to newer versions, copy the new exact numbers
    # here too rather than reopening the range.
    import subprocess
    subprocess.run(
        ["pip", "install", "-q",
         "unsloth", "unsloth_zoo",
         "transformers==4.57.6",
         "trl==0.24.0",
         "peft==0.20.0",
         "accelerate>=0.34.1"],
        check=True,
    )
else:
    !pip install --no-deps bitsandbytes "accelerate>=0.34.1" xformers==0.0.29.post3 peft==0.20.0 trl==0.24.0 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" huggingface_hub hf_transfer transformers==4.57.6
    !pip install --no-deps unsloth

In [ ]:
import transformers, peft
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)

## 1. Config

In [ ]:
import os, json, re, random, hashlib
import numpy as np
import torch

SEED = 3407
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MAX_SEQ_LENGTH = 4096   # tokenizer truncation cap; BASE_MODEL_PATH is resolved below

import glob as _glob

COMP = "/kaggle/input/goedel-machines-x-iitm-clinical-llm-challenge"

# --- Knowledge base -------------------------------------------------------
# Auto-discovered: the KB dataset's folder slug depends on what it was titled
# (rag-dset / rag_dataset / rag-dataset ...), and hardcoding one has broken this
# notebook before. Falls back to any .jsonl in a rag-ish dataset folder.
_kb = (sorted(_glob.glob("/kaggle/input/*/master_dataset_rag.jsonl"))
       or sorted(_glob.glob("/kaggle/input/*rag*/*.jsonl")))
KB_PATH = _kb[0] if _kb else None
assert KB_PATH, ("No KB jsonl found under /kaggle/input/*/. Attach the RAG dataset "
                 "(the one holding master_dataset_rag.jsonl).")
print(f"KB: {KB_PATH}")
if len(_kb) > 1:
    print(f"  ⚠️  multiple candidates, using the first: {_kb}")

# --- Adapter source -------------------------------------------------------
# Auto-discovered rather than hardcoded: the adapter lands under whatever slug
# the training notebook was saved as, and hardcoding that has broken this
# notebook once already. Attach the training notebook's output as an input.
# Recursive, not just one level -- a hardcoded top-level glob missed the
# adapter once already even with the notebook attached (Kaggle sometimes
# nests notebook-output inputs deeper than /kaggle/input/<slug>/<file>).
_adapters = sorted(_glob.glob("/kaggle/input/**/loramodel_final", recursive=True))
LORA_PATH_HUB = "mouryesh/phi35-clinical-lora"
LORA_PATH = _adapters[0] if _adapters else LORA_PATH_HUB
TOKENIZER_PATH = LORA_PATH          # training notebook saves both together
print(f"adapter: {LORA_PATH}" + ("" if _adapters else "   (no local copy found -- trying the Hub)"))
if len(_adapters) > 1:
    print(f"  ⚠️  multiple adapters found, using the first: {_adapters}")

# --- Base model + embedder -------------------------------------------------
# Prefer a locally-attached copy if one happens to be present (faster, no
# network round-trip); otherwise fall back to the HF Hub repo id, which needs
# Internet=On. Searches broadly (Kaggle Models attach under
# /kaggle/input/<model-slug>/<variation>/, an extra nesting level vs Datasets)
# rather than assuming one exact path.
def _find_hf_model(keywords, min_files=3):
    hits = []
    for cfg in _glob.glob("/kaggle/input/**/config.json", recursive=True):
        d = os.path.dirname(cfg)
        if all(k.lower() in d.lower() for k in keywords):
            n_files = len([f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))])
            if n_files >= min_files:
                hits.append(d)
    return sorted(set(hits))

BASE_MODEL_HUB   = "unsloth/Phi-3.5-mini-instruct"
MINILM_HUB       = "sentence-transformers/all-MiniLM-L6-v2"

_base_hits = _find_hf_model(["phi-3.5"]) or _find_hf_model(["phi3.5"]) or _find_hf_model(["phi-3", "5"])
BASE_MODEL_PATH = _base_hits[0] if _base_hits else BASE_MODEL_HUB

_minilm_hits = _find_hf_model(["minilm"])
MINILM_PATH = _minilm_hits[0] if _minilm_hits else MINILM_HUB

print(f"\nbase model : {BASE_MODEL_PATH}" + ("" if _base_hits else "   (no local copy -- downloading from the Hub)"))
print(f"embedder   : {MINILM_PATH}" + ("" if _minilm_hits else "   (no local copy -- downloading from the Hub)"))

# --- Pre-flight download, with a real error instead of a cryptic one later --
# `AutoModelForCausalLM.from_pretrained` on an incomplete/corrupted HF cache
# can fail deep inside transformers with `AttributeError: 'NoneType' object
# has no attribute 'endswith'` -- checkpoint file resolution silently landing
# on None, rather than a clear "download failed" error. Downloading explicitly
# here with huggingface_hub, force_download=True to bypass any stale/partial
# cache from an earlier interrupted attempt, surfaces the REAL cause (network,
# 401, repo not found, disk space) instead of that opaque AttributeError three
# layers into someone else's library. Also cheaply confirms Hub connectivity is
# real -- `pip install` reaching PyPI doesn't guarantee huggingface.co is
# reachable too; they're different domains and can be gated differently.
from huggingface_hub import snapshot_download

def _ensure_local(path_or_repo, hub_id, label):
    if os.path.isdir(path_or_repo):
        return path_or_repo   # already local (Kaggle-attached copy)
    try:
        local = snapshot_download(repo_id=hub_id, force_download=True)
        print(f"✅ {label} downloaded -> {local}")
        return local
    except Exception as e:
        print(f"\n{'!' * 78}")
        print(f"❌ {label} download FAILED: {type(e).__name__}: {e}")
        print(f"   This means huggingface.co isn't reachable (even if pypi.org is --")
        print(f"   they're different domains and can be gated separately), or the repo")
        print(f"   id is wrong, or there's a Hub-side issue. Check Settings -> Internet")
        print(f"   is really On, or attach {label} locally via Add Input -> Models.")
        print(f"{'!' * 78}")
        raise

BASE_MODEL_PATH = _ensure_local(BASE_MODEL_PATH, BASE_MODEL_HUB, "base model")
MINILM_PATH     = _ensure_local(MINILM_PATH, MINILM_HUB, "embedder")

# --- Eval set -------------------------------------------------------------
# The competition's own dev files are 10-100 rows. At n=10 a truly 75%-accurate
# model carries a +-27 percentage-point 95% CI, so those files cannot separate
# base from fine-tuned -- any "improvement" measured on them would be noise.
# The training notebook writes a ~3k held-out split (never trained on, same
# distribution, +-1.5pp), so prefer that whenever it is available.
_held = sorted(_glob.glob("/kaggle/input/**/held_out_val.jsonl", recursive=True))
if _held:
    DEV_SETS = {"held_out": _held[0]}
    print(f"eval set: {_held[0]}  (held-out split -- statistically usable)")
else:
    DEV_SETS = {
        "dev_medium": f"{COMP}/sample_test_medium.jsonl",
        "dev_hard":   f"{COMP}/sample_test_hard.jsonl",
    }
    print("⚠️  held_out_val.jsonl not found -- falling back to the 10-row competition")
    print("   dev files. Results from those are NOT statistically meaningful; attach")
    print("   the finetune-lora notebook output to get the real eval set.")

TEST_SETS = {   # no answers -- submission only
    "test_easy":   f"{COMP}/test_easy.jsonl",
    "test_medium": f"{COMP}/test_medium.jsonl",
}

# What the LoRA actually saw -- must match finetune-lora.ipynb's TRAIN_SOURCES,
# or the contamination check silently measures the wrong thing. Found by
# filename rather than assumed folder slug, since a hardcoded path missed both
# files once already even with the datasets attached (Kaggle's mount folder
# name isn't always the display name, e.g. "mediam shufled" -> some other slug).
def _find_by_filename(filename):
    hits = sorted(_glob.glob(f"/kaggle/input/**/{filename}", recursive=True))
    return hits[0] if hits else None

TRAIN_SOURCES = [p for p in [
    _find_by_filename("randomized_output.jsonl"),
    _find_by_filename("shuffled_input_medum.jsonl"),
] if p]
print(f"train sources found: {TRAIN_SOURCES or '❌ none -- attach 30k-dataset and mediam shufled'}")

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Load data + inspect the KB schema

The retrieval quality fix depends on which fields the KB actually has — check the printout below before trusting the `KB_*` settings in the next cell.

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"  skipping malformed line {i+1} in {path}")
    return rows

kb_rows = load_jsonl(KB_PATH)
print(f"KB rows: {len(kb_rows):,}")
print("KB fields:", sorted(kb_rows[0].keys()))
print("\n--- sample KB row ---")
print(json.dumps(kb_rows[0], indent=2)[:1500])

for name, path in {**DEV_SETS, **TEST_SETS}.items():
    if os.path.exists(path):
        rows = load_jsonl(path)
        print(f"\n{name}: {len(rows):,} rows | fields: {sorted(rows[0].keys())} | has answer: {'answer' in rows[0]}")
    else:
        print(f"\n{name}: MISSING at {path}")

## 3. Contamination check

If eval questions appear in the training data or the retrieval KB, the accuracy numbers are inflated and the RAG gain is an artifact. Report these numbers in the writeup either way.

In [ ]:
def norm_q(s):
    return re.sub(r"\s+", " ", str(s)).strip().lower()

def qset(rows, field="question"):
    return {norm_q(r[field]) for r in rows if field in r}

HELD_OUT_MODE = "held_out" in DEV_SETS

if HELD_OUT_MODE:
    # IMPORTANT: do NOT compare the held-out split against the raw source files.
    # held_out_val.jsonl is a 10% slice OF those files, taken after dedup and
    # never shown to the trainer. The source files still contain those rows, so
    # that comparison would report ~100% overlap -- a false alarm, not leakage.
    # The split is disjoint from the TRAINING rows by construction, which is the
    # property that actually matters.
    print("=" * 78)
    print("Eval set = the training notebook's held-out split.")
    print("Disjoint from the training rows BY CONSTRUCTION (split taken after dedup;")
    print("those rows were never fed to the trainer), so there is nothing to test here.")
    print()
    print("Comparing it to the raw source files would show ~100% overlap -- the files")
    print("still contain the held-out rows -- so that comparison is deliberately NOT")
    print("made. It would look alarming and mean nothing.")
    print("=" * 78)

    # Surface the check that IS meaningful: it ran upstream in the training
    # notebook, comparing the training data against the competition eval files.
    _prior = sorted(_glob.glob("/kaggle/input/*/results/contamination.json"))
    contamination = {"mode": "held_out_split_disjoint_by_construction"}
    if _prior:
        with open(_prior[0]) as f:
            prior = json.load(f)
        contamination["upstream"] = prior
        print("\nUpstream check (training data vs competition eval files), from")
        print(f"{_prior[0]}:")
        for k, v in prior.items():
            if isinstance(v, dict) and "pct" in v:
                print(f"   {k:30s} {v['overlap']:>5,}/{v['n_eval']:<6,} = {v['pct']:5.2f}%")
    else:
        print("\n⚠️  upstream contamination.json not found in the attached notebook output.")

    train_qs = set()
    TRAIN_AVAILABLE = True

else:
    TRAIN_FILES = [p for p in TRAIN_SOURCES if os.path.exists(p)]
    TRAIN_AVAILABLE = bool(TRAIN_FILES)
    if not TRAIN_AVAILABLE:
        print("!" * 78)
        print("TRAINING SOURCES NOT ATTACHED -- overlap numbers below are MEANINGLESS")
        print("(they read 0% because there is nothing to compare against, not because")
        print("the data is clean). Attach '30k-dataset' and 'mediam shufled'.")
        for p in TRAIN_SOURCES:
            print(f"   {'ok  ' if os.path.exists(p) else 'MISS'} {p}")
        print("!" * 78)
    else:
        print(f"Training sources: {[os.path.basename(f) for f in TRAIN_FILES]}")

    train_rows = []
    for tf in TRAIN_FILES:
        train_rows += load_jsonl(tf)
    train_qs = qset(train_rows)
    contamination = {}

# KB entries are instruction-style; the instruction often *is* the question.
kb_qs = {norm_q(r.get("instruction", "")) for r in kb_rows}
kb_qs.discard("")

print(f"\nKB instructions: {len(kb_qs):,}"
      + (f"   train questions: {len(train_qs):,}" if not HELD_OUT_MODE else ""))

# The KB check matters in BOTH modes and is the one that can quietly inflate the
# RAG rows: if an eval question sits verbatim in the retrieval knowledge base,
# retrieval hands the model the answer and "RAG helped" becomes an artifact.
for name, path in DEV_SETS.items():
    if not os.path.exists(path):
        continue
    ev = qset(load_jsonl(path))
    in_kb = len(ev & kb_qs)
    entry = {"n_eval": len(ev),
             "overlap_with_kb": in_kb,
             "overlap_with_kb_pct": round(100 * in_kb / max(len(ev), 1), 2)}
    print(f"\n{name}: n={len(ev):,}")
    print(f"   in KB   : {in_kb:,} ({entry['overlap_with_kb_pct']:.2f}%)"
          + ("   ⚠️  RAG rows are inflated -- report this" if entry["overlap_with_kb_pct"] > 1 else ""))

    if HELD_OUT_MODE:
        entry["overlap_with_train"] = "n/a -- disjoint by construction"
        print("   in TRAIN: n/a (held-out split, never trained on)")
    else:
        in_train = len(ev & train_qs)
        entry["train_set_attached"] = TRAIN_AVAILABLE
        entry["overlap_with_train"] = in_train
        entry["overlap_with_train_pct"] = round(100 * in_train / max(len(ev), 1), 2)
        print(f"   in TRAIN: {in_train:,} ({entry['overlap_with_train_pct']:.2f}%)")
    contamination[name] = entry

with open(f"{RESULTS_DIR}/contamination.json", "w") as f:
    json.dump(contamination, f, indent=2)
print(f"\nsaved -> {RESULTS_DIR}/contamination.json")

## 4. Build the retrieval index

Two changes from the original: the injected context includes the KB row's
**answer/output** field (previously only the question-like `instruction` was injected,
so the model got a similar question with no information in it), and FAISS uses cosine
similarity (`IndexFlatIP` on normalised vectors), which is what MiniLM is trained for.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# --- confirm these against the schema printed in section 2 ---
KB_EMBED_FIELD    = "instruction"                    # what the question is matched against
KB_CONTEXT_FIELDS = ["instruction", "output"]        # what actually gets injected as context

available = set(kb_rows[0].keys())
KB_CONTEXT_FIELDS = [f for f in KB_CONTEXT_FIELDS if f in available]
assert KB_EMBED_FIELD in available, f"{KB_EMBED_FIELD} not in KB fields {sorted(available)}"
if len(KB_CONTEXT_FIELDS) < 2:
    print(f"⚠️  Only injecting {KB_CONTEXT_FIELDS}. If the KB has an answer/response/output-like "
          f"field under another name, add it -- context without an answer is near-useless.")

def kb_context_text(row):
    parts = []
    for f in KB_CONTEXT_FIELDS:
        v = str(row.get(f, "")).strip()
        if v:
            parts.append(v)
    return "\n".join(parts)

kb_embed_texts   = [str(r.get(KB_EMBED_FIELD, "")).strip() for r in kb_rows]
kb_context_texts = [kb_context_text(r) for r in kb_rows]

embedder = SentenceTransformer(MINILM_PATH)   # local path if found, else downloads MINILM_HUB
kb_embeddings = embedder.encode(
    kb_embed_texts, convert_to_numpy=True, show_progress_bar=True,
    batch_size=256, normalize_embeddings=True,
).astype("float32")

index = faiss.IndexFlatIP(kb_embeddings.shape[1])   # cosine, since vectors are normalised
index.add(kb_embeddings)
print(f"✅ FAISS index: {index.ntotal:,} vectors, dim {kb_embeddings.shape[1]}")

def retrieve(question, k=1):
    q = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q, k)
    return [(kb_context_texts[i], float(s)) for i, s in zip(idx[0], scores[0]) if i >= 0]

demo = retrieve("A 55-year-old man presents with crushing chest pain radiating to the left arm.", k=2)
for i, (txt, sc) in enumerate(demo):
    print(f"\n--- hit {i+1} (cos={sc:.3f}) ---\n{txt[:400]}")

## 5. Load the model

The LoRA adapter is loaded once and toggled with `disable_adapter()`, so base and
fine-tuned configs share a single 4-bit model in memory — both fit on one T4.

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel
from transformers import AutoTokenizer
from contextlib import nullcontext

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL_PATH,   # local path if found, else downloads BASE_MODEL_HUB
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)

# Tokenizer saved alongside training, so special tokens match. Non-fatal if missing.
try:
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
    print(f"✅ tokenizer from {TOKENIZER_PATH}")
except Exception as e:
    print(f"⚠️  could not load {TOKENIZER_PATH}, using base tokenizer instead: {e}")

# The final comparison needs the adapter -- fail loudly rather than silently
# reporting a base-only "comparison" that has nothing to compare.
try:
    model = PeftModel.from_pretrained(model, LORA_PATH)
    LORA_AVAILABLE = True
    print(f"✅ LoRA adapter loaded from {LORA_PATH}")
except Exception as e:
    LORA_AVAILABLE = False
    raise RuntimeError(
        f"Could not load the fine-tuned adapter from {LORA_PATH}: {e}\n"
        f"Fix: attach the finetune-lora notebook's output (it saves 'loramodel_final'), "
        f"or confirm {LORA_PATH_HUB} exists on the Hub and Internet=On."
    ) from e

FastLanguageModel.for_inference(model)
model.eval()

def adapter_ctx(use_lora):
    """LoRA on -> no-op context; LoRA off -> disable_adapter()."""
    return nullcontext() if use_lora else model.disable_adapter()

## 6. Prompt construction

This reproduces the training template byte-for-byte up to `<|assistant|>`. The only delta when RAG is on is an inserted `### Context:` block, so the base-vs-full-pipeline comparison isolates fine-tuning + retrieval rather than being confounded by a prompt-format change.

In [ ]:
SYSTEM_MSG = ("You are preparing medical students for board examinations (USMLE, COMLEX). "
              "Provide accurate, evidence-based answers consistent with current medical standards.")

USER_HEAD = ("This is a medical board examination question. Apply your knowledge of clinical "
             "medicine, basic sciences, and current guidelines.")

USER_TAIL = ("Choose the letter corresponding to the BEST answer based on current medical "
             "knowledge and clinical practice guidelines.")

def format_options(options):
    if isinstance(options, dict):
        return "\n".join(f"{k}. {v}" for k, v in sorted(options.items()))
    return "\n".join(f"{chr(65+i)}. {v}" for i, v in enumerate(options))

def build_prompt(question, options, context=None):
    body = USER_HEAD + "\n\n"
    if context:
        body += f"### Context:\n{context}\n\n"
    body += f"### Question:\n{question.strip()}\n\n"
    body += f"### Options:\n{format_options(options)}\n\n"
    body += USER_TAIL
    return (f"<|system|>\n{SYSTEM_MSG}<|end|>\n"
            f"<|user|>\n{body}<|end|>\n"
            f"<|assistant|>\n")

print(build_prompt(
    "A 55-year-old man presents with crushing chest pain. Most likely diagnosis?",
    {"A": "Myocardial infarction", "B": "GERD", "C": "Costochondritis", "D": "Panic attack"},
    context="Chest pain radiating to the left arm with diaphoresis suggests acute MI.",
))

## 7. Scoring

`logit` mode reads the log-prob of each option letter at the first assistant token — deterministic and immune to parsing failures. `generate` mode is kept for comparison and now decodes **only** the newly generated tokens.

In [ ]:
def build_letter_token_ids(tokenizer, sample_prompt):
    """First token id produced for each letter immediately after the prompt."""
    base = tokenizer(sample_prompt, add_special_tokens=False)["input_ids"]
    ids = {}
    for L in "ABCD":
        full = tokenizer(sample_prompt + L, add_special_tokens=False)["input_ids"]
        if full[:len(base)] == base and len(full) > len(base):
            ids[L] = full[len(base)]
        else:  # tokenizer merged across the boundary -- fall back
            ids[L] = tokenizer.encode(L, add_special_tokens=False)[0]
    assert len(set(ids.values())) == 4, f"letter tokens collide: {ids}"
    return ids

_sample = build_prompt("q?", {"A": "a", "B": "b", "C": "c", "D": "d"})
LETTER_IDS = build_letter_token_ids(tokenizer, _sample)
print("letter -> token id:", LETTER_IDS)


@torch.no_grad()
def predict_logit(prompt):
    enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=MAX_SEQ_LENGTH).to(model.device)
    logits = model(**enc).logits[0, -1].float()
    logprobs = torch.log_softmax(logits, dim=-1)
    scores = {L: logprobs[tid].item() for L, tid in LETTER_IDS.items()}
    return max(scores, key=scores.get), scores


def extract_answer_letter(text):
    """Pull an option letter out of *generated* text (never the prompt)."""
    t = text.strip()
    if not t:
        return None
    m = re.match(r"\(?\s*([A-Da-d])\s*[\)\.\:,]?(?:\s|$)", t)
    if m:
        return m.group(1).upper()
    m = re.search(r"\b([A-Da-d])\b", t)
    return m.group(1).upper() if m else None


@torch.no_grad()
def predict_generate(prompt, max_new_tokens=8):
    enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=MAX_SEQ_LENGTH).to(model.device)
    out = model.generate(
        input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
        max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    # THE BUG: previously decoded out[0] (prompt + completion), so the regex matched
    # the "A." in the options block. Slice off the prompt first.
    gen = out[0][enc["input_ids"].shape[1]:]
    text = tokenizer.decode(gen, skip_special_tokens=True)
    return extract_answer_letter(text), text


# --- self-tests: catch the old bug before spending GPU time ---
assert extract_answer_letter("B. Myocardial infarction") == "B"
assert extract_answer_letter(" C") == "C"
assert extract_answer_letter("(D)") == "D"
assert extract_answer_letter("The answer is A.") == "A"
assert extract_answer_letter("") is None
_old_style = _sample + "B. b"          # prompt + completion, as the old code decoded it
assert re.search(r"\b([A-D])\b", _old_style).group(1) == "A", "sanity: old code always read A"
print("✅ extractor self-tests pass (and the old prompt-inclusive decode does return 'A')")

## 8. The evaluation harness

One function, parameterised by `use_lora` / `use_rag` / `k`, so the base run and the full-pipeline run share every line of code — the only difference between the two numbers we report is these flags.

In [ ]:
import pandas as pd
from collections import Counter
from tqdm.auto import tqdm

def evaluate(dataset_name, path, use_lora=True, use_rag=False, k=1,
             score_mode="logit", limit=None, save=True, verbose_examples=0):
    if use_lora and not LORA_AVAILABLE:
        raise RuntimeError("use_lora=True but no adapter is loaded.")

    rows = load_jsonl(path)
    if limit:
        rows = rows[:limit]

    cfg = f"{'lora' if use_lora else 'base'}_{('rag' + str(k)) if use_rag else 'norag'}_{score_mode}"
    tag = f"{dataset_name}__{cfg}"

    preds, n_correct, n_scored, shown = [], 0, 0, 0

    with adapter_ctx(use_lora):
        for ex in tqdm(rows, desc=tag):
            question = str(ex["question"]).strip()
            options  = ex["options"]

            context, hits = None, []
            if use_rag:
                hits = retrieve(question, k=k)
                if hits:
                    context = "\n\n".join(h[0] for h in hits)

            prompt = build_prompt(question, options, context)

            if score_mode == "logit":
                pred, scores = predict_logit(prompt)
                raw = json.dumps({kk: round(v, 3) for kk, v in scores.items()})
            else:
                pred, raw = predict_generate(prompt)

            rec = {
                "id": ex.get("id", None),
                "question": question,
                "predicted_option": pred if pred else "NA",
                "raw": raw,
                "retrieval_score": round(hits[0][1], 4) if hits else None,
            }
            if "answer" in ex:
                gt = str(ex["answer"]).strip().upper()
                rec["gt"] = gt
                rec["correct"] = (pred == gt)
                n_correct += rec["correct"]
                n_scored += 1

                if verbose_examples and shown < verbose_examples and use_rag and not rec["correct"]:
                    shown += 1
                    print(f"\n[miss] pred={pred} gt={gt}\nQ: {question[:200]}\nCTX: {str(context)[:300]}")

            preds.append(rec)

    acc = (100 * n_correct / n_scored) if n_scored else None
    result = {
        "dataset": dataset_name, "config": cfg, "use_lora": use_lora, "use_rag": use_rag,
        "k": k if use_rag else 0, "score_mode": score_mode,
        "n": len(rows), "n_scored": n_scored, "n_correct": n_correct,
        "accuracy": round(acc, 2) if acc is not None else None,
    }

    if save:
        pd.DataFrame(preds).to_csv(f"{RESULTS_DIR}/preds__{tag}.csv", index=False)
        with open(f"{RESULTS_DIR}/metrics__{tag}.json", "w") as f:
            json.dump(result, f, indent=2)

    print(f"\n{tag}: accuracy={result['accuracy']}%  ({n_correct}/{n_scored})"
          f"   split={dict(Counter(p['predicted_option'] for p in preds))}")
    return result, preds

print("✅ harness ready")

## 9. Smoke test

Run a handful of examples first. If the option split here is degenerate (all one letter) something is still wrong — stop and fix before launching the full matrix.

In [ ]:
# Whatever DEV_SETS actually contains -- "held_out" when the training notebook's
# split was found, or the fallback "dev_medium"/"dev_hard" competition files
# otherwise (section 1 decides which). Hardcoding one name here broke this
# exact cell once already.
_smoke_name, _smoke_path = next(iter(DEV_SETS.items()))
print(f"smoke-testing on: {_smoke_name} ({_smoke_path})")
_r, _p = evaluate(_smoke_name, _smoke_path, use_lora=True, use_rag=False,
                  score_mode="logit", limit=25, save=False)

## 10. Pick k for retrieval

A small sweep to choose how many passages the final pipeline retrieves — this is tuning the one system we're reporting, not a comparison against anything old. `LIMIT` keeps this cheap; it does not affect the final numbers in §12, which always run on the full dev sets.

In [ ]:
LIMIT = 300      # subset size for tuning only
K_VALUES = [1, 3, 5]

k_results = []
for ds_name, ds_path in DEV_SETS.items():
    if not os.path.exists(ds_path):
        continue
    for k in K_VALUES:
        res, _ = evaluate(ds_name, ds_path, use_lora=True, use_rag=True, k=k,
                          score_mode="logit", limit=LIMIT, save=False)
        k_results.append(res)

k_df = pd.DataFrame(k_results)
k_df.to_csv(f"{RESULTS_DIR}/k_sweep.csv", index=False)
print(k_df[["dataset", "k", "accuracy"]].to_string(index=False))

BEST_K = int(k_df.loc[k_df.groupby("dataset")["accuracy"].idxmax(), "k"].mode().iloc[0])
print(f"\n➡️  BEST_K = {BEST_K} (mode of per-dataset best; override manually below if you disagree)")

## 11. Final comparison: base model vs. fine-tuned + RAG

The two numbers that matter. Same harness, same eval sets, same prompt template — the only difference between the rows is whether the adapter and retrieval are switched on.

In [ ]:
# LIMIT=None for the numbers going in the writeup. Uses BEST_K from the sweep above.
FINAL_LIMIT = None

configs = [
    ("base",              dict(use_lora=False, use_rag=False, k=0)),
    ("fine-tuned + RAG",  dict(use_lora=True,  use_rag=True,  k=BEST_K)),
]

final_results = []
for ds_name, ds_path in DEV_SETS.items():
    if not os.path.exists(ds_path):
        print(f"skipping missing {ds_name}")
        continue
    for label, cfg in configs:
        res, _ = evaluate(ds_name, ds_path, score_mode="logit", limit=FINAL_LIMIT, **cfg)
        res["label"] = label
        final_results.append(res)

results_df = pd.DataFrame(final_results)
results_df.to_csv(f"{RESULTS_DIR}/final_comparison.csv", index=False)
results_df

## 12. Summary table

In [ ]:
pivot = results_df.pivot_table(index="label", columns="dataset", values="accuracy")
pivot = pivot.reindex(["base", "fine-tuned + RAG"])

print("Accuracy (%) by configuration\n")
print(pivot.to_string())
pivot.to_csv(f"{RESULTS_DIR}/summary_table.csv")

for ds in results_df.dataset.unique():
    row = results_df[results_df.dataset == ds]
    base_acc = row[row.label == "base"].accuracy.iloc[0]
    full_acc = row[row.label == "fine-tuned + RAG"].accuracy.iloc[0]
    print(f"\n{ds}: base {base_acc:.2f}%  ->  fine-tuned+RAG {full_acc:.2f}%  ({full_acc - base_acc:+.2f} pts)")

## 13. Qualitative: where the full pipeline changed the answer

Cases worth putting in the writeup — concrete evidence the fine-tune + retrieval combination is doing something, not just noise.

In [ ]:
def compare_final(ds_name, k=BEST_K):
    a = pd.read_csv(f"{RESULTS_DIR}/preds__{ds_name}__base_norag_logit.csv")
    b = pd.read_csv(f"{RESULTS_DIR}/preds__{ds_name}__lora_rag{k}_logit.csv")
    m = a.merge(b, on="question", suffixes=("_base", "_full"))
    if "correct_base" not in m.columns:
        print("no ground truth in this split")
        return None
    fixed  = m[(~m.correct_base) & (m.correct_full)]
    broken = m[(m.correct_base) & (~m.correct_full)]
    print(f"{ds_name}: pipeline fixed {len(fixed)}, pipeline broke {len(broken)}, net {len(fixed)-len(broken):+d}")
    for title, frame in (("FIXED", fixed), ("BROKEN", broken)):
        for r in frame.head(3).itertuples():
            print(f"\n--- {title} ---\nQ: {r.question[:250]}")
            print(f"base={r.predicted_option_base}  full={r.predicted_option_full}  gt={r.gt_base}")
            print(f"retrieval cos={r.retrieval_score_full}")
    return fixed, broken

for ds in DEV_SETS:
    if os.path.exists(f"{RESULTS_DIR}/preds__{ds}__base_norag_logit.csv"):
        compare_final(ds)

## 14. Submission

Uses the fine-tuned + RAG pipeline at the k chosen in §10.

In [ ]:
BEST = dict(use_lora=True, use_rag=True, k=BEST_K)

submission = []
for ds_name, ds_path in TEST_SETS.items():
    if not os.path.exists(ds_path):
        print(f"skipping missing {ds_name}")
        continue
    _, preds = evaluate(ds_name, ds_path, score_mode="logit", limit=None, **BEST)
    submission += [{"id": p["id"], "predicted_option": p["predicted_option"]} for p in preds]

sub_df = pd.DataFrame(submission)
sub_df.to_csv("final_submission.csv", index=False)
print(f"\n✅ wrote final_submission.csv with {len(sub_df):,} rows")

from collections import Counter
print("\nOption split:", dict(sorted(Counter(sub_df.predicted_option).items())))
if sub_df.predicted_option.value_counts(normalize=True).iloc[0] > 0.5:
    print("⚠️  one option dominates >50% -- likely still a scoring bug, do not submit.")